# Only Train Pipeline Notebook (Self-Contained)

This notebook runs **SFT (domain checkpoints) + expert fusion + MGPO on RL dataset** end-to-end.


## 1. Imports and Global Settings

In [ ]:
import ast
import copy
import json
import math
import os
import re
import signal
import subprocess
import sys
import tempfile
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim import Optimizer
from torch.optim.lr_scheduler import LambdaLR
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    PreTrainedTokenizerBase,
    get_cosine_schedule_with_warmup,
)

try:
    import resource  # POSIX-only
except Exception:
    resource = None


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


## 2. Prompt Tokens and Formatting

In [ ]:
QUESTION_TOKEN = "<|question|>"
EXAMPLE_TOKEN = "<|example|>"
REASONING_TOKEN = "<|reasoning|>"
CODE_TOKEN = "<|code|>"

SPECIAL_TOKENS = [
    QUESTION_TOKEN,
    EXAMPLE_TOKEN,
    REASONING_TOKEN,
    CODE_TOKEN,
]


class PromptFormatter:
    @staticmethod
    def format_input_prompt(question: str, example: str) -> str:
        return (
            f"{QUESTION_TOKEN}\n"
            f"{question.strip()}\n"
            f"{EXAMPLE_TOKEN}\n"
            f"{example.strip()}\n"
            f"{REASONING_TOKEN}\n"
        )

    @staticmethod
    def format_output_target(reasoning: str, solution: str) -> str:
        return (
            f"{reasoning.strip()}\n\n"
            f"{CODE_TOKEN}\n"
            f"{solution.strip()}"
        )

    @staticmethod
    def format_sft_sample(question: str, example: str, reasoning: str, solution: str) -> str:
        return (
            PromptFormatter.format_input_prompt(question=question, example=example)
            + PromptFormatter.format_output_target(reasoning=reasoning, solution=solution)
        )

    @staticmethod
    def format_generation_prompt(question: str, example: str) -> str:
        return PromptFormatter.format_input_prompt(question=question, example=example)


def parse_domain_csvs(raw_value: str) -> Dict[str, str]:
    items = [x.strip() for x in raw_value.split(",") if x.strip()]
    result: Dict[str, str] = {}
    for item in items:
        if ":" not in item:
            raise ValueError(f"Invalid domain spec '{item}'. Use domain:path.csv format.")
        domain, path = item.split(":", maxsplit=1)
        result[domain.strip()] = path.strip()
    if not result:
        raise ValueError("No domain CSVs parsed from domain_csvs.")
    return result


## 3. Model Factory

In [ ]:
@dataclass
class ModelConfig:
    model_name: str = "gpt2"


class ModelFactory:
    def __init__(self, config: ModelConfig) -> None:
        self.config = config

    def load_model(self, model_name_or_path: str):
        return AutoModelForCausalLM.from_pretrained(model_name_or_path)

    def build(self):
        tokenizer = AutoTokenizer.from_pretrained(self.config.model_name)
        model = self.load_model(self.config.model_name)

        if tokenizer.pad_token is None:
            if tokenizer.eos_token is not None:
                tokenizer.pad_token = tokenizer.eos_token
            else:
                tokenizer.add_special_tokens({"pad_token": "[PAD]"})

        tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})
        model.config.pad_token_id = tokenizer.pad_token_id
        model.resize_token_embeddings(len(tokenizer))

        return model, tokenizer


## 4. Dataset and Dataloaders

In [ ]:
@dataclass
class DataConfig:
    max_length: int = 768
    train_batch_size: int = 2
    eval_batch_size: int = 2
    num_workers: int = 0
    reasoning_weight: float = 1.0
    code_weight: float = 1.0
    spec_weight: float = 1.0


class CausalLMDataset(Dataset):
    def __init__(
        self,
        rows: List[dict],
        tokenizer,
        max_length: int,
        reasoning_weight: float = 1.0,
        code_weight: float = 1.0,
        spec_weight: float = 1.0,
    ) -> None:
        self.rows = rows
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.reasoning_weight = float(reasoning_weight)
        self.code_weight = float(code_weight)
        self.spec_weight = float(spec_weight)
        self.eos_token_id = tokenizer.eos_token_id

    def __len__(self) -> int:
        return len(self.rows)

    def _encode_target_with_weights(self, row: dict) -> Tuple[List[int], List[float]]:
        reasoning_text = f"{row['reasoning'].strip()}\n\n"
        code_marker_text = f"{CODE_TOKEN}\n"
        code_text = row["solution"].strip()

        reasoning_ids = self.tokenizer(reasoning_text, add_special_tokens=False)["input_ids"]
        code_marker_ids = self.tokenizer(code_marker_text, add_special_tokens=False)["input_ids"]
        code_ids = self.tokenizer(code_text, add_special_tokens=False)["input_ids"]

        target_ids = reasoning_ids + code_marker_ids + code_ids
        target_weights = (
            [self.reasoning_weight] * len(reasoning_ids)
            + [self.spec_weight] * len(code_marker_ids)
            + [self.code_weight] * len(code_ids)
        )

        if self.eos_token_id is not None:
            target_ids.append(self.eos_token_id)
            target_weights.append(self.code_weight)

        return target_ids, target_weights

    def __getitem__(self, idx: int):
        row = self.rows[idx]

        prompt_text = PromptFormatter.format_input_prompt(
            question=row["question"],
            example=row.get("example", ""),
        )
        prompt_ids = self.tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
        target_ids, target_weights = self._encode_target_with_weights(row)

        input_ids = prompt_ids + target_ids
        labels = input_ids.copy()
        token_weights = [0.0] * len(prompt_ids) + target_weights

        input_ids = input_ids[: self.max_length]
        labels = labels[: self.max_length]
        token_weights = token_weights[: self.max_length]

        for i in range(min(len(prompt_ids), len(labels))):
            labels[i] = -100

        input_ids = torch.tensor(input_ids, dtype=torch.long)
        attention_mask = torch.ones_like(input_ids)
        labels = torch.tensor(labels, dtype=torch.long)
        token_weights = torch.tensor(token_weights, dtype=torch.float32)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "token_weights": token_weights,
        }


class CausalLMDataCollator:
    def __init__(self, tokenizer) -> None:
        self.tokenizer = tokenizer

    def __call__(self, batch):
        input_ids = [item["input_ids"] for item in batch]
        attention_masks = [item["attention_mask"] for item in batch]
        labels = [item["labels"] for item in batch]
        token_weights = [item["token_weights"] for item in batch]

        input_ids = torch.nn.utils.rnn.pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id,
        )
        attention_mask = torch.nn.utils.rnn.pad_sequence(
            attention_masks,
            batch_first=True,
            padding_value=0,
        )
        labels = torch.nn.utils.rnn.pad_sequence(
            labels,
            batch_first=True,
            padding_value=-100,
        )
        token_weights = torch.nn.utils.rnn.pad_sequence(
            token_weights,
            batch_first=True,
            padding_value=0.0,
        )

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "token_weights": token_weights,
        }


class ReasoningDataModule:
    def __init__(self, config: DataConfig, tokenizer) -> None:
        self.config = config
        self.tokenizer = tokenizer
        self.collator = CausalLMDataCollator(tokenizer=tokenizer)

    @staticmethod
    def _normalize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
        rename_map = {"reasonings": "reasoning", "solutions": "solution"}
        df = df.rename(columns=rename_map)

        if "example" not in df.columns:
            df["example"] = ""
        if "reasoning" not in df.columns:
            df["reasoning"] = ""

        required_columns = ["question", "example", "reasoning", "solution"]
        missing = [col for col in required_columns if col not in df.columns]
        if missing:
            raise ValueError(
                f"CSV does not contain required columns: {missing}. Found columns: {list(df.columns)}"
            )

        return df[required_columns].fillna("").astype(str)

    def _load_rows(self, csv_path: str) -> List[dict]:
        df = pd.read_csv(csv_path)
        df = self._normalize_dataframe(df)
        return df.to_dict(orient="records")

    def load_rows_from_csv(self, csv_path: str) -> List[dict]:
        return self._load_rows(csv_path)

    def build_dataloaders_from_rows(self, train_rows: List[dict], val_rows: List[dict]) -> Tuple[DataLoader, DataLoader]:
        train_dataset = CausalLMDataset(
            rows=train_rows,
            tokenizer=self.tokenizer,
            max_length=self.config.max_length,
            reasoning_weight=self.config.reasoning_weight,
            code_weight=self.config.code_weight,
            spec_weight=self.config.spec_weight,
        )
        val_dataset = CausalLMDataset(
            rows=val_rows,
            tokenizer=self.tokenizer,
            max_length=self.config.max_length,
            reasoning_weight=self.config.reasoning_weight,
            code_weight=self.config.code_weight,
            spec_weight=self.config.spec_weight,
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=self.config.train_batch_size,
            shuffle=True,
            num_workers=self.config.num_workers,
            collate_fn=self.collator,
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=self.config.eval_batch_size,
            shuffle=False,
            num_workers=self.config.num_workers,
            collate_fn=self.collator,
        )
        return train_loader, val_loader


def build_data_module(
    tokenizer: PreTrainedTokenizerBase,
    max_length: int,
    train_batch_size: int,
    eval_batch_size: int,
    num_workers: int,
    reasoning_weight: float = 1.0,
    code_weight: float = 1.0,
    spec_weight: float = 1.0,
) -> ReasoningDataModule:
    data_config = DataConfig(
        max_length=max_length,
        train_batch_size=train_batch_size,
        eval_batch_size=eval_batch_size,
        num_workers=num_workers,
        reasoning_weight=reasoning_weight,
        code_weight=code_weight,
        spec_weight=spec_weight,
    )
    return ReasoningDataModule(data_config, tokenizer)


## 5. Scheduler Helper

In [ ]:
def build_cosine_scheduler_with_warmup(
    optimizer: Optimizer,
    num_warmup_steps: int,
    num_training_steps: int,
):
    num_training_steps = max(1, int(num_training_steps))
    num_warmup_steps = max(0, min(int(num_warmup_steps), num_training_steps))

    try:
        return get_cosine_schedule_with_warmup(
            optimizer=optimizer,
            num_warmup_steps=num_warmup_steps,
            num_training_steps=num_training_steps,
        )
    except Exception:
        def lr_lambda(current_step: int) -> float:
            if current_step < num_warmup_steps:
                return float(current_step) / float(max(1, num_warmup_steps))
            if num_training_steps == num_warmup_steps:
                return 0.0
            progress = float(current_step - num_warmup_steps) / float(num_training_steps - num_warmup_steps)
            progress = min(max(progress, 0.0), 1.0)
            return 0.5 * (1.0 + math.cos(math.pi * progress))

        return LambdaLR(optimizer, lr_lambda=lr_lambda)


## 6. Safe Verifiers (Binary + Graded)

In [ ]:
import ast
import json
import os
import re
import signal
import subprocess
import sys
import tempfile
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

try:
    import resource  # POSIX-only
except Exception:  # pragma: no cover
    resource = None


EXECUTION_TIMEOUT_SECONDS = 2.0
CPU_LIMIT_SECONDS = 2
MEMORY_LIMIT_BYTES = 256 * 1024 * 1024
FILE_SIZE_LIMIT_BYTES = 1 * 1024 * 1024
MAX_STDOUT_CHARS = 50_000
MAX_STDERR_CHARS = 20_000
MAX_RESULT_REPR_CHARS = 20_000
MAX_PIPE_CHARS = 300_000

FORBIDDEN_IMPORT_ROOTS = {
    "os",
    "sys",
    "subprocess",
    "socket",
    "shutil",
    "pathlib",
    "multiprocessing",
    "signal",
}
FORBIDDEN_CALL_NAMES = {
    "exec",
    "eval",
    "compile",
    "__import__",
}

ERROR_SYNTAX = "syntax_error"
ERROR_RUNTIME = "runtime_error"
ERROR_TIMEOUT = "timeout"
ERROR_MEMORY = "memory_error"
ERROR_UNSAFE = "unsafe_code"
ERROR_OK = "ok"

FLOAT_ATOL = 1e-8
FLOAT_RTOL = 1e-8


@dataclass
class ExecutionResult:
    status: str
    result: Any = None
    stdout: str = ""
    stderr: str = ""
    detail: str = ""


_RUNNER_CODE = r"""
import ast
import contextlib
import io
import json
import traceback
import sys


class LimitedBuffer(io.StringIO):
    def __init__(self, max_chars: int) -> None:
        super().__init__()
        self.max_chars = max_chars
        self.overflow = False

    def write(self, s):
        if not isinstance(s, str):
            s = str(s)
        current = self.tell()
        remaining = self.max_chars - current
        if remaining <= 0:
            self.overflow = True
            return len(s)
        if len(s) > remaining:
            super().write(s[:remaining])
            self.overflow = True
            return len(s)
        return super().write(s)


def _safe_repr(value, max_chars: int):
    text = repr(value)
    if len(text) <= max_chars:
        return text, False
    return text[:max_chars], True


def _emit(payload):
    sys.stdout.write(json.dumps(payload, ensure_ascii=False))
    sys.stdout.write("\n")
    sys.stdout.flush()


def _parse_call_input(input_text: str):
    raw = (input_text or "").strip()
    if not raw:
        return tuple(), {}

    try:
        parsed = ast.parse(f"_f({raw})", mode="eval")
        call_node = parsed.body
        if isinstance(call_node, ast.Call):
            args = [ast.literal_eval(arg) for arg in call_node.args]
            kwargs = {}
            for kw in call_node.keywords:
                if kw.arg is None:
                    raise ValueError("Unsupported **kwargs input.")
                kwargs[kw.arg] = ast.literal_eval(kw.value)
            return tuple(args), kwargs
    except Exception:
        pass

    try:
        val = ast.literal_eval(raw)
        if isinstance(val, tuple):
            return val, {}
        return (val,), {}
    except Exception:
        return (raw,), {}


def _run_function_mode(compiled, payload):
    function_name = payload.get("function_name") or ""
    input_text = payload.get("input_text") or ""
    max_output_chars = int(payload.get("max_output_chars", 50000))
    max_result_repr_chars = int(payload.get("max_result_repr_chars", 20000))

    namespace = {"__name__": "__candidate__"}
    stdout_buffer = LimitedBuffer(max_output_chars)
    stderr_buffer = LimitedBuffer(max_output_chars)
    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        exec(compiled, namespace, namespace)
        fn = namespace.get(function_name)
        if fn is None or not callable(fn):
            raise ValueError(f"Function '{function_name}' not found or not callable.")
        args, kwargs = _parse_call_input(input_text)
        result = fn(*args, **kwargs)

    result_repr, result_truncated = _safe_repr(result, max_result_repr_chars)
    return {
        "status": "ok",
        "result_repr": result_repr,
        "result_truncated": result_truncated,
        "stdout": stdout_buffer.getvalue(),
        "stderr": stderr_buffer.getvalue(),
        "stdout_overflow": stdout_buffer.overflow,
        "stderr_overflow": stderr_buffer.overflow,
    }


def _run_script_mode(compiled, payload):
    stdin_text = payload.get("stdin_text") or ""
    max_output_chars = int(payload.get("max_output_chars", 50000))
    namespace = {"__name__": "__main__"}
    stdout_buffer = LimitedBuffer(max_output_chars)
    stderr_buffer = LimitedBuffer(max_output_chars)
    fake_stdin = io.StringIO(stdin_text)

    with contextlib.redirect_stdout(stdout_buffer), contextlib.redirect_stderr(stderr_buffer):
        old_stdin = sys.stdin
        try:
            sys.stdin = fake_stdin
            exec(compiled, namespace, namespace)
        finally:
            sys.stdin = old_stdin

    return {
        "status": "ok",
        "stdout": stdout_buffer.getvalue(),
        "stderr": stderr_buffer.getvalue(),
        "stdout_overflow": stdout_buffer.overflow,
        "stderr_overflow": stderr_buffer.overflow,
    }


def main():
    payload_path = sys.argv[1]
    with open(payload_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    candidate_path = payload["candidate_path"]
    mode = payload.get("mode") or "function"
    with open(candidate_path, "r", encoding="utf-8") as f:
        candidate_code = f.read()

    try:
        compiled = compile(candidate_code, candidate_path, "exec")
    except SyntaxError as e:
        _emit({"status": "syntax_error", "error": str(e)})
        return
    except Exception as e:
        _emit({"status": "runtime_error", "error": str(e)})
        return

    try:
        if mode == "function":
            _emit(_run_function_mode(compiled, payload))
            return
        if mode == "script":
            _emit(_run_script_mode(compiled, payload))
            return
        _emit({"status": "runtime_error", "error": f"Unknown mode '{mode}'"})
    except MemoryError:
        _emit({"status": "memory_error", "error": "MemoryError"})
    except SyntaxError as e:
        _emit({"status": "syntax_error", "error": str(e)})
    except Exception as e:
        _emit(
            {
                "status": "runtime_error",
                "error": str(e),
                "traceback": traceback.format_exc(limit=6),
            }
        )


if __name__ == "__main__":
    main()
"""


def _truncate_text(text: str, max_chars: int) -> str:
    if len(text) <= max_chars:
        return text
    return text[:max_chars]


def normalize_text(text: str) -> str:
    return " ".join(text.replace("```python", "```").replace("```", "").split()).strip().lower()


def extract_code_block(text: str) -> str:
    match = re.search(r"```(?:python)?\s*(.*?)```", text or "", flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return (text or "").strip()


def extract_first_function_name(code: str) -> Optional[str]:
    try:
        tree = ast.parse(code)
    except Exception:
        return None
    for node in tree.body:
        if isinstance(node, ast.FunctionDef):
            return node.name
    return None


def parse_examples(example_text: str) -> List[Tuple[str, str]]:
    text = (example_text or "").replace("\r\n", "\n").strip()
    if not text:
        return []

    pairs: List[Tuple[str, str]] = []
    try:
        # Main flexible parser for blocks:
        # Input: ... ; Output: ...
        pattern = r"Input:\s*(.*?)\s*(?:;)?\s*Output:\s*(.*?)(?=\n\s*Input:|$)"
        for inp, out in re.findall(pattern, text, flags=re.DOTALL | re.IGNORECASE):
            inp_clean = inp.strip()
            out_clean = out.strip()
            if inp_clean and out_clean:
                pairs.append((inp_clean, out_clean))
        if pairs:
            return pairs
    except Exception:
        pass

    # Fallback line parser.
    try:
        current_input: List[str] = []
        current_output: List[str] = []
        mode: Optional[str] = None
        lines = text.split("\n")
        for line in lines:
            stripped = line.strip()
            lower = stripped.lower()
            if lower.startswith("input:"):
                if current_input or current_output:
                    inp = "\n".join(current_input).strip()
                    out = "\n".join(current_output).strip()
                    if inp and out:
                        pairs.append((inp, out))
                current_input = [line.split(":", 1)[1].strip()]
                current_output = []
                mode = "input"
                continue
            if lower.startswith("output:"):
                current_output = [line.split(":", 1)[1].strip()]
                mode = "output"
                continue
            if mode == "input":
                current_input.append(line)
            elif mode == "output":
                current_output.append(line)

        inp = "\n".join(current_input).strip()
        out = "\n".join(current_output).strip()
        if inp and out:
            pairs.append((inp, out))
    except Exception:
        return []

    return pairs


def parse_call_input(input_text: str) -> Tuple[Tuple, Dict]:
    raw = (input_text or "").strip()
    if not raw:
        return tuple(), {}

    # Robust parse via synthetic call expression.
    try:
        parsed = ast.parse(f"_f({raw})", mode="eval")
        call_node = parsed.body
        if isinstance(call_node, ast.Call):
            args = [ast.literal_eval(arg) for arg in call_node.args]
            kwargs = {}
            for kw in call_node.keywords:
                if kw.arg is None:
                    raise ValueError("Unsupported **kwargs style input.")
                kwargs[kw.arg] = ast.literal_eval(kw.value)
            return tuple(args), kwargs
    except Exception:
        pass

    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, tuple):
            return parsed, {}
        return (parsed,), {}
    except Exception:
        pass

    parts = [x for x in re.split(r"[,\s]+", raw) if x]
    if parts:
        converted = []
        numeric = True
        for part in parts:
            try:
                if "." in part:
                    converted.append(float(part))
                else:
                    converted.append(int(part))
            except Exception:
                numeric = False
                break
        if numeric:
            return tuple(converted), {}

    return (raw,), {}


def parse_expected_output(output_text: str):
    try:
        return ast.literal_eval((output_text or "").strip())
    except Exception:
        return (output_text or "").strip()


def _is_number(value: Any) -> bool:
    return isinstance(value, (int, float)) and not isinstance(value, bool)


def _numbers_close(a: float, b: float) -> bool:
    diff = abs(a - b)
    limit = FLOAT_ATOL + FLOAT_RTOL * max(abs(a), abs(b))
    return diff <= limit


def _iterable_like(value: Any) -> bool:
    return isinstance(value, (list, tuple))


def results_equal(predicted, expected) -> bool:
    if _is_number(predicted) and _is_number(expected):
        return _numbers_close(float(predicted), float(expected))

    if isinstance(predicted, str) and isinstance(expected, str):
        return normalize_text(predicted) == normalize_text(expected)

    if _iterable_like(predicted) and _iterable_like(expected):
        if len(predicted) != len(expected):
            return False
        return all(results_equal(p, e) for p, e in zip(predicted, expected))

    if isinstance(predicted, dict) and isinstance(expected, dict):
        if predicted.keys() != expected.keys():
            return False
        return all(results_equal(predicted[key], expected[key]) for key in predicted.keys())

    if isinstance(predicted, (set, frozenset)) and isinstance(expected, (set, frozenset)):
        if len(predicted) != len(expected):
            return False
        pred_list = list(predicted)
        exp_list = list(expected)
        used = [False] * len(exp_list)
        for pred_item in pred_list:
            matched = False
            for i, exp_item in enumerate(exp_list):
                if used[i]:
                    continue
                if results_equal(pred_item, exp_item):
                    used[i] = True
                    matched = True
                    break
            if not matched:
                return False
        return True

    if predicted == expected:
        return True

    return normalize_text(str(predicted)) == normalize_text(str(expected))


def _root_module_name(module_name: str) -> str:
    return (module_name or "").split(".", 1)[0]


def _called_name(func_node: ast.AST) -> Optional[str]:
    if isinstance(func_node, ast.Name):
        return func_node.id
    if isinstance(func_node, ast.Attribute):
        return func_node.attr
    return None


def _analyze_code_safety(code: str) -> Tuple[bool, str]:
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return False, ERROR_SYNTAX
    except Exception:
        return False, ERROR_RUNTIME

    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if _root_module_name(alias.name) in FORBIDDEN_IMPORT_ROOTS:
                    return False, ERROR_UNSAFE
        elif isinstance(node, ast.ImportFrom):
            if _root_module_name(node.module or "") in FORBIDDEN_IMPORT_ROOTS:
                return False, ERROR_UNSAFE
        elif isinstance(node, ast.Call):
            called = _called_name(node.func)
            if called in FORBIDDEN_CALL_NAMES:
                return False, ERROR_UNSAFE

    return True, ERROR_OK


def _build_preexec_fn():
    if resource is None or os.name != "posix":
        return None

    def _set_limits():
        try:
            resource.setrlimit(resource.RLIMIT_CPU, (CPU_LIMIT_SECONDS, CPU_LIMIT_SECONDS))
        except Exception:
            pass
        try:
            resource.setrlimit(resource.RLIMIT_AS, (MEMORY_LIMIT_BYTES, MEMORY_LIMIT_BYTES))
        except Exception:
            pass
        try:
            resource.setrlimit(resource.RLIMIT_FSIZE, (FILE_SIZE_LIMIT_BYTES, FILE_SIZE_LIMIT_BYTES))
        except Exception:
            pass

    return _set_limits


def _build_safe_env() -> Dict[str, str]:
    env: Dict[str, str] = {
        "PYTHONIOENCODING": "utf-8",
        "PYTHONUNBUFFERED": "1",
    }
    for key in ("PATH", "SYSTEMROOT", "WINDIR", "HOME", "TMPDIR", "TEMP", "TMP", "LANG", "LC_ALL"):
        if key in os.environ:
            env[key] = os.environ[key]
    return env


def _format_call_input(args: Tuple, kwargs: Dict) -> str:
    if kwargs:
        parts = [f"{k}={repr(v)}" for k, v in kwargs.items()]
        if args:
            parts = [repr(x) for x in args] + parts
        return ", ".join(parts)
    if len(args) == 0:
        return ""
    if len(args) == 1:
        return repr(args[0])
    return ", ".join(repr(x) for x in args)


def _deserialize_result_repr(result_repr: str):
    try:
        return ast.literal_eval(result_repr)
    except Exception:
        return result_repr


def _parse_runner_payload(stdout_text: str) -> Optional[Dict[str, Any]]:
    for line in reversed(stdout_text.splitlines()):
        raw = line.strip()
        if not raw:
            continue
        try:
            payload = json.loads(raw)
            if isinstance(payload, dict):
                return payload
        except Exception:
            continue
    return None


def _map_status(status: str) -> str:
    if status in {ERROR_OK, ERROR_SYNTAX, ERROR_RUNTIME, ERROR_TIMEOUT, ERROR_MEMORY, ERROR_UNSAFE}:
        return status
    return ERROR_RUNTIME


def _run_candidate_subprocess(
    code: str,
    mode: str,
    *,
    input_text: str = "",
    function_name: str = "",
) -> ExecutionResult:
    try:
        with tempfile.TemporaryDirectory() as tmp_dir:
            candidate_path = os.path.join(tmp_dir, "candidate.py")
            runner_path = os.path.join(tmp_dir, "_runner.py")
            payload_path = os.path.join(tmp_dir, "_payload.json")

            with open(candidate_path, "w", encoding="utf-8") as f:
                f.write(code)
            with open(runner_path, "w", encoding="utf-8") as f:
                f.write(_RUNNER_CODE)

            payload = {
                "candidate_path": candidate_path,
                "mode": mode,
                "function_name": function_name,
                "input_text": input_text,
                "stdin_text": input_text,
                "max_output_chars": MAX_STDOUT_CHARS,
                "max_result_repr_chars": MAX_RESULT_REPR_CHARS,
            }
            with open(payload_path, "w", encoding="utf-8") as f:
                json.dump(payload, f, ensure_ascii=False)

            run_kwargs: Dict[str, Any] = {
                "cwd": tmp_dir,
                "capture_output": True,
                "text": True,
                "timeout": EXECUTION_TIMEOUT_SECONDS,
                "env": _build_safe_env(),
                "close_fds": True,
            }
            preexec_fn = _build_preexec_fn()
            if preexec_fn is not None:
                run_kwargs["preexec_fn"] = preexec_fn

            try:
                completed = subprocess.run(
                    [sys.executable, runner_path, payload_path],
                    **run_kwargs,
                )
            except subprocess.TimeoutExpired:
                return ExecutionResult(status=ERROR_TIMEOUT, detail="execution_timeout")
    except Exception as e:
        return ExecutionResult(status=ERROR_RUNTIME, detail=f"subprocess_setup_error: {e}")

    stdout_text = _truncate_text(completed.stdout or "", MAX_PIPE_CHARS)
    stderr_text = _truncate_text(completed.stderr or "", MAX_PIPE_CHARS)
    payload = _parse_runner_payload(stdout_text)

    if payload is None:
        if completed.returncode < 0 and resource is not None:
            sig = -completed.returncode
            if sig in {getattr(signal, "SIGKILL", -9), getattr(signal, "SIGSEGV", -11)}:
                return ExecutionResult(status=ERROR_MEMORY, stdout=stdout_text, stderr=stderr_text)
            if sig == getattr(signal, "SIGXCPU", -24):
                return ExecutionResult(status=ERROR_TIMEOUT, stdout=stdout_text, stderr=stderr_text)
        return ExecutionResult(
            status=ERROR_RUNTIME,
            stdout=stdout_text,
            stderr=stderr_text,
            detail="runner_payload_parse_failed",
        )

    status = _map_status(str(payload.get("status", ERROR_RUNTIME)))
    combined_stdout = _truncate_text(str(payload.get("stdout", "")), MAX_STDOUT_CHARS)
    combined_stderr = _truncate_text(str(payload.get("stderr", "")), MAX_STDERR_CHARS)

    if status != ERROR_OK:
        return ExecutionResult(
            status=status,
            stdout=combined_stdout or stdout_text,
            stderr=combined_stderr or stderr_text,
            detail=str(payload.get("error", "")),
        )

    if payload.get("stdout_overflow") or payload.get("stderr_overflow") or payload.get("result_truncated"):
        return ExecutionResult(
            status=ERROR_RUNTIME,
            stdout=combined_stdout,
            stderr=combined_stderr,
            detail="output_limit_exceeded",
        )

    if mode == "function":
        result = _deserialize_result_repr(str(payload.get("result_repr", "")))
    else:
        result = combined_stdout

    return ExecutionResult(
        status=ERROR_OK,
        result=result,
        stdout=combined_stdout,
        stderr=combined_stderr,
    )


def run_function(code: str, function_name: str, args: Tuple, kwargs: Dict):
    input_text = _format_call_input(args, kwargs)
    result = _run_candidate_subprocess(code, "function", input_text=input_text, function_name=function_name)
    if result.status != ERROR_OK:
        raise ValueError(result.status)
    return result.result


def run_script_with_stdin(code: str, stdin_text: str):
    result = _run_candidate_subprocess(code, "script", input_text=stdin_text)
    if result.status != ERROR_OK:
        raise ValueError(result.status)
    return str(result.result)


def _strict_interface_requires_reference_name(example_text: str, reference_name: Optional[str]) -> bool:
    if not reference_name:
        return False
    pattern = rf"\b{re.escape(reference_name)}\s*\("
    return re.search(pattern, example_text or "", flags=re.IGNORECASE) is not None


def _choose_function_name(
    pred_name: Optional[str],
    ref_name: Optional[str],
    example_text: str,
) -> Optional[str]:
    if _strict_interface_requires_reference_name(example_text, ref_name):
        return ref_name
    if pred_name:
        return pred_name
    return ref_name


def _run_function_tests(
    pred_code: str,
    tests: List[Tuple[str, str]],
    function_name: Optional[str],
) -> Tuple[bool, int, int]:
    if not function_name:
        return False, 0, len(tests)

    passed = 0
    for input_text, output_text in tests:
        expected = parse_expected_output(output_text)
        result = _run_candidate_subprocess(
            pred_code,
            "function",
            input_text=input_text,
            function_name=function_name,
        )
        if result.status != ERROR_OK:
            return False, 0, len(tests)
        if results_equal(result.result, expected):
            passed += 1
    return True, passed, len(tests)


def _run_script_tests(
    pred_code: str,
    tests: List[Tuple[str, str]],
) -> Tuple[bool, int, int]:
    passed = 0
    for input_text, output_text in tests:
        expected = parse_expected_output(output_text)
        result = _run_candidate_subprocess(pred_code, "script", input_text=input_text)
        if result.status != ERROR_OK:
            return False, 0, len(tests)
        if results_equal(result.result, expected):
            passed += 1
    return True, passed, len(tests)


class BinaryCodeVerifier:
    def __call__(self, prediction: str, reference: str, example: str) -> bool:
        try:
            pred_code = extract_code_block(prediction)
            safe, status = _analyze_code_safety(pred_code)
            if not safe:
                return False
            if status == ERROR_SYNTAX:
                return False

            tests = parse_examples(example)
            if not tests:
                return False

            ref_code = extract_code_block(reference)
            pred_name = extract_first_function_name(pred_code)
            ref_name = extract_first_function_name(ref_code)

            fn_name = _choose_function_name(pred_name, ref_name, example)
            function_ok, function_passed, function_total = _run_function_tests(pred_code, tests, fn_name)
            if function_ok and function_passed == function_total:
                return True

            script_ok, script_passed, script_total = _run_script_tests(pred_code, tests)
            return script_ok and script_passed == script_total
        except Exception:
            return False


def _count_passed_tests(pred_code: str, reference: str, example: str) -> Tuple[int, int, bool]:
    tests = parse_examples(example)
    if not tests:
        return 0, 0, True

    ref_code = extract_code_block(reference)
    pred_name = extract_first_function_name(pred_code)
    ref_name = extract_first_function_name(ref_code)
    fn_name = _choose_function_name(pred_name, ref_name, example)

    function_ok, function_passed, function_total = _run_function_tests(pred_code, tests, fn_name)
    if function_ok:
        return function_passed, function_total, True

    script_ok, script_passed, script_total = _run_script_tests(pred_code, tests)
    if script_ok:
        return script_passed, script_total, True
    return 0, len(tests), False


class GradedCodeVerifier:
    def __call__(self, prediction: str, reference: str, example: str) -> float:
        try:
            pred_code = extract_code_block(prediction)
            safe, status = _analyze_code_safety(pred_code)
            if not safe:
                return 0.0
            if status == ERROR_SYNTAX:
                return 0.0

            passed, total, executed = _count_passed_tests(pred_code, reference, example)
            if total <= 0:
                return 0.1
            if not executed:
                return 0.0
            return 0.1 + 0.9 * (float(passed) / float(total))
        except Exception:
            return 0.0


## 7. Train Pipeline (SFT + Domain Experts + Fusion + MGPO)


In [ ]:
import csv


@dataclass
class GRPOConfig:
    learning_rate: float
    weight_decay: float
    batch_size: int
    group_size: int
    max_new_tokens: int
    steps: int
    max_length: int
    clip_eps: float
    kl_beta: float


class GRPOTrainer:
    def __init__(
        self,
        model,
        tokenizer,
        train_rows: List[dict],
        device: torch.device,
        config: GRPOConfig,
        verifier,
    ) -> None:
        self.model = model
        self.tokenizer = tokenizer
        self.train_rows = train_rows
        self.device = device
        self.config = config
        self.verifier = verifier

        self.model.to(self.device)
        self.optimizer = AdamW(self.model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)

        self.reference_model = copy.deepcopy(self.model).eval().to(self.device)
        for p in self.reference_model.parameters():
            p.requires_grad_(False)

    @staticmethod
    def _normalize_reward(value: Any) -> float:
        if isinstance(value, bool):
            return 1.0 if value else 0.0
        try:
            numeric = float(value)
        except Exception:
            return 0.0
        if numeric < 0.0:
            return 0.0
        if numeric > 1.0:
            return 1.0
        return numeric

    def compute_group_weight(self, p_empirical: float) -> float:
        return 1.0

    def _safe_prompt_max_length(self) -> int:
        prompt_max = max(8, int(self.config.max_length))

        model_max_positions = getattr(self.model.config, 'n_positions', None)
        if model_max_positions is None:
            model_max_positions = getattr(self.model.config, 'max_position_embeddings', None)
        if isinstance(model_max_positions, int) and model_max_positions > 0:
            headroom = max(8, int(model_max_positions) - int(self.config.max_new_tokens) - 1)
            prompt_max = min(prompt_max, headroom)

        tokenizer_max = getattr(self.tokenizer, 'model_max_length', None)
        if isinstance(tokenizer_max, int) and 0 < tokenizer_max < 1_000_000:
            prompt_max = min(prompt_max, tokenizer_max)

        return max(8, int(prompt_max))

    def _sanitize_prompt_ids(self, prompt_ids: torch.Tensor) -> torch.Tensor:
        vocab_size = int(self.model.get_input_embeddings().weight.shape[0])
        if vocab_size <= 0 or prompt_ids.numel() == 0:
            return prompt_ids

        min_id = int(prompt_ids.min().item())
        max_id = int(prompt_ids.max().item())
        if min_id < 0 or max_id >= vocab_size:
            prompt_ids = prompt_ids.clamp(min=0, max=vocab_size - 1)
        return prompt_ids

    def _extract_response_ids(self, prompt_ids: torch.Tensor, generated_ids: torch.Tensor) -> torch.Tensor:
        return generated_ids[0][prompt_ids.shape[1]:]

    def _sequence_logprob(self, model, prompt_ids: torch.Tensor, response_ids: torch.Tensor) -> torch.Tensor:
        if response_ids.numel() == 0:
            return torch.tensor(0.0, device=self.device)

        full_ids = torch.cat([prompt_ids.squeeze(0), response_ids], dim=0).unsqueeze(0)
        attention_mask = torch.ones_like(full_ids, device=self.device)
        outputs = model(input_ids=full_ids, attention_mask=attention_mask)
        logits = outputs.logits[:, :-1, :]
        targets = full_ids[:, 1:]
        token_log_probs = torch.log_softmax(logits, dim=-1).gather(dim=-1, index=targets.unsqueeze(-1)).squeeze(-1)
        prompt_len = prompt_ids.shape[1]
        return token_log_probs[:, max(prompt_len - 1, 0):].sum()

    @torch.no_grad()
    def _collect_group_rollouts(self, row: dict):
        prompt = PromptFormatter.format_generation_prompt(row['question'], row.get('example', ''))
        safe_prompt_max = self._safe_prompt_max_length()
        encoded = self.tokenizer(prompt, return_tensors='pt', truncation=True, max_length=safe_prompt_max)
        prompt_ids = self._sanitize_prompt_ids(encoded['input_ids']).to(self.device)
        prompt_mask = encoded['attention_mask'].to(self.device)

        samples = []
        self.model.eval()
        for _ in range(self.config.group_size):
            try:
                generated = self.model.generate(
                    input_ids=prompt_ids,
                    attention_mask=prompt_mask,
                    do_sample=True,
                    top_p=0.95,
                    temperature=0.9,
                    max_new_tokens=self.config.max_new_tokens,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
            except RuntimeError as e:
                message = str(e)
                if 'indexSelectSmallIndex' in message or 'device-side assert' in message:
                    print('[MGPO] generation skipped due to CUDA index error')
                    return []
                raise
            response_ids = self._extract_response_ids(prompt_ids, generated).detach()
            prediction = self.tokenizer.decode(response_ids, skip_special_tokens=True)
            reward = self._normalize_reward(self.verifier(prediction, row.get('solution', ''), row.get('example', '')))

            old_logprob = self._sequence_logprob(self.model, prompt_ids, response_ids).detach()
            ref_logprob = self._sequence_logprob(self.reference_model, prompt_ids, response_ids).detach()
            samples.append(
                {
                    'prompt_ids': prompt_ids.detach(),
                    'response_ids': response_ids,
                    'reward': reward,
                    'old_logprob': old_logprob,
                    'ref_logprob': ref_logprob,
                }
            )
        return samples

    def train_step(self, batch_rows: List[dict]) -> Dict[str, float]:
        self.model.train()
        all_losses = []
        all_weights = []
        all_p = []

        rollouts_per_row = [self._collect_group_rollouts(row) for row in batch_rows]

        for row_rollouts in rollouts_per_row:
            if not row_rollouts:
                continue
            rewards = torch.tensor([x['reward'] for x in row_rollouts], dtype=torch.float32, device=self.device)
            p_empirical = float(rewards.mean().item())
            all_p.append(p_empirical)

            mean = rewards.mean()
            std = rewards.std(unbiased=False)
            advantages = (rewards - mean) / (std + 1e-6)

            w_me = self.compute_group_weight(p_empirical)
            all_weights.append(w_me)
            weighted_advantages = advantages * w_me

            row_losses = []
            for idx, sample in enumerate(row_rollouts):
                prompt_ids = sample['prompt_ids'].to(self.device)
                response_ids = sample['response_ids'].to(self.device)
                old_logprob = sample['old_logprob'].to(self.device)
                ref_logprob = sample['ref_logprob'].to(self.device)
                adv = weighted_advantages[idx]

                current_logprob = self._sequence_logprob(self.model, prompt_ids, response_ids)
                ratio = torch.exp(torch.clamp(current_logprob - old_logprob, min=-20.0, max=20.0))
                clipped_ratio = torch.clamp(ratio, 1.0 - self.config.clip_eps, 1.0 + self.config.clip_eps)
                surrogate = torch.min(ratio * adv, clipped_ratio * adv)
                policy_loss = -surrogate

                kl_proxy = torch.relu(current_logprob - ref_logprob)
                row_losses.append(policy_loss + self.config.kl_beta * kl_proxy)

            all_losses.append(torch.stack(row_losses).mean())

        loss = torch.stack(all_losses).mean() if all_losses else torch.tensor(0.0, device=self.device)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
        self.optimizer.step()

        return {
            'loss': float(loss.item()),
            'avg_weight': float(np.mean(all_weights)) if all_weights else 1.0,
            'avg_p': float(np.mean(all_p)) if all_p else 0.0,
        }

    def train(self) -> List[Dict[str, float]]:
        history = []
        if not self.train_rows:
            return history

        progress = tqdm(range(1, self.config.steps + 1), desc='mgpo/grpo steps')
        for step in progress:
            batch_rows = random.sample(self.train_rows, k=min(self.config.batch_size, len(self.train_rows)))
            metrics = self.train_step(batch_rows)
            metrics['step'] = step
            history.append(metrics)

            if step % 10 == 0:
                print(
                    f"[MGPO] step={step} loss={metrics['loss']:.4f} "
                    f"avg_p={metrics['avg_p']:.4f} avg_w={metrics['avg_weight']:.4f}"
                )
        return history


class MGPOTrainer(GRPOTrainer):
    def __init__(
        self,
        model,
        tokenizer,
        train_rows: List[dict],
        device: torch.device,
        config: GRPOConfig,
        verifier,
        lambda_me: float,
    ) -> None:
        super().__init__(model, tokenizer, train_rows, device, config, verifier)
        self.lambda_me = float(lambda_me)

    def compute_group_weight(self, p_empirical: float) -> float:
        eps = 1e-6
        p = min(max(float(p_empirical), eps), 1.0 - eps)
        p0 = 0.5
        d_me = p * math.log(p / p0) + (1.0 - p) * math.log((1.0 - p) / (1.0 - p0))
        return float(math.exp(-self.lambda_me * d_me))


@dataclass
class TrainConfig:
    model_name: str = 'gpt2'

    sft_train_csv_path: str = 'data/reasoning_dataset.csv'
    domain_csvs: str = 'alg:data/alg_tasks.csv,math:data/math_tasks.csv,struct:data/struct_tasks.csv'
    rl_train_csv_path: str = 'data/rl_tasks_from_llm_2kplus.csv'
    output_dir: str = 'artifacts_train'
    rl_init_model_path: str = ''

    seed: int = 42
    max_length: int = 768
    train_batch_size: int = 2
    eval_batch_size: int = 2
    num_workers: int = 0

    reasoning_weight: float = 1.0
    code_weight: float = 1.0
    spec_weight: float = 1.0

    sft_epochs: int = 2
    sft_learning_rate: float = 5e-5
    sft_weight_decay: float = 0.01
    sft_warmup_ratio: float = 0.05
    sft_grad_accum_steps: int = 1
    max_grad_norm: float = 1.0
    sft_eval_every_epochs: int = 3
    sft_eval_samples_per_domain: int = 20
    sft_save_every_epochs: int = 1
    sft_resume: bool = True

    pass_k: int = 3
    pass_k_samples: int = 8
    eval_max_new_tokens: int = 256
    eval_temperature: float = 0.8
    eval_top_p: float = 0.95
    eval_max_rows: int = 0

    rl_steps: int = 100
    rl_learning_rate: float = 1e-5
    rl_weight_decay: float = 0.0
    rl_batch_size: int = 4
    rl_group_size: int = 8
    rl_clip_eps: float = 0.2
    rl_kl_beta: float = 0.0
    rl_lambda_me: float = 4.0
    rl_max_new_tokens: int = 128


class SFTEngine:
    def __init__(
        self,
        model,
        tokenizer,
        config: TrainConfig,
        device: torch.device,
    ) -> None:
        self.model = model
        self.tokenizer = tokenizer
        self.config = config
        self.device = device

    def _extract_response_ids(self, prompt_ids: torch.Tensor, generated_ids: torch.Tensor) -> torch.Tensor:
        return generated_ids[0][prompt_ids.shape[1]:]

    def _compute_weighted_loss(self, logits: torch.Tensor, labels: torch.Tensor, token_weights: torch.Tensor) -> torch.Tensor:
        aligned_logits = logits[:, :-1, :]
        aligned_labels = labels[:, 1:]
        aligned_weights = token_weights[:, 1:]

        vocab_size = aligned_logits.shape[-1]
        token_loss = F.cross_entropy(
            aligned_logits.reshape(-1, vocab_size),
            aligned_labels.reshape(-1),
            ignore_index=-100,
            reduction='none',
        ).reshape_as(aligned_labels)

        valid_mask = (aligned_labels != -100).float()
        aligned_weights = aligned_weights * valid_mask
        denom = aligned_weights.sum().clamp(min=1e-8)
        return (token_loss * aligned_weights).sum() / denom

    def train_epoch(self, loader, optimizer, scheduler, accum_steps: int, max_grad_norm: float) -> float:
        self.model.train()
        loss_sum = torch.tensor(0.0, device=self.device)
        step_count = torch.tensor(0.0, device=self.device)

        optimizer.zero_grad()
        progress = tqdm(loader, desc='sft train', leave=False)
        for micro_step, batch in enumerate(progress, start=1):
            batch = {k: v.to(self.device) for k, v in batch.items()}
            token_weights = batch.pop('token_weights', None)

            outputs = self.model(**batch)
            if token_weights is None:
                loss = outputs.loss
            else:
                loss = self._compute_weighted_loss(outputs.logits, batch['labels'], token_weights)

            (loss / accum_steps).backward()
            is_update_step = (micro_step % accum_steps == 0) or (micro_step == len(loader))
            if is_update_step:
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            loss_sum = loss_sum + loss.detach()
            step_count = step_count + 1.0

        return float(loss_sum.item() / max(1.0, step_count.item()))

    @torch.no_grad()
    def evaluate_pass_at_k(self, rows: List[dict], verifier, k: int, n_samples: int, max_rows: int) -> float:
        self.model.eval()
        sample_rows = rows[:max_rows] if max_rows > 0 else rows
        if not sample_rows:
            return 0.0

        scores: List[float] = []
        progress = tqdm(sample_rows, desc='pass@k eval', leave=False)
        for row in progress:
            prompt = PromptFormatter.format_generation_prompt(
                question=row['question'],
                example=row.get('example', ''),
            )
            prompt_max_length = self.config.max_length
            model_max_positions = getattr(self.model.config, 'n_positions', None)
            if model_max_positions is None:
                model_max_positions = getattr(self.model.config, 'max_position_embeddings', None)
            if isinstance(model_max_positions, int) and model_max_positions > 0:
                safe_prompt = max(8, int(model_max_positions) - int(self.config.eval_max_new_tokens) - 1)
                prompt_max_length = min(prompt_max_length, safe_prompt)

            encoded = self.tokenizer(
                prompt,
                return_tensors='pt',
                truncation=True,
                max_length=prompt_max_length,
            )
            input_ids = encoded['input_ids']
            vocab_size = int(self.model.get_input_embeddings().weight.shape[0])
            if input_ids.numel() > 0:
                input_ids = input_ids.clamp(min=0, max=vocab_size - 1)
            input_ids = input_ids.to(self.device)
            attention_mask = encoded['attention_mask'].to(self.device)

            correct = 0
            for _ in range(n_samples):
                generated_ids = self.model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    do_sample=True,
                    top_p=self.config.eval_top_p,
                    temperature=self.config.eval_temperature,
                    max_new_tokens=self.config.eval_max_new_tokens,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
                response_ids = self._extract_response_ids(input_ids, generated_ids)
                completion = self.tokenizer.decode(response_ids, skip_special_tokens=True)
                if verifier(completion, row.get('solution', ''), row.get('example', '')):
                    correct += 1

            scores.append(estimate_pass_at_k(n_samples, correct, k))

        return float(np.mean(scores)) if scores else 0.0


def estimate_pass_at_k(n: int, c: int, k: int) -> float:
    if c <= 0:
        return 0.0
    if n - c < k:
        return 1.0
    indices = np.arange(n - c + 1, n + 1)
    return float(1.0 - np.prod(1.0 - k / indices))


def _normalize_pairs_text(pairs: List[Tuple[str, str]]) -> str:
    return '\n'.join(f"Input: {inp}; Output: {out}" for inp, out in pairs)


def _try_parse_pairs(text: str) -> List[Tuple[str, str]]:
    raw = (text or '').strip()
    if not raw:
        return []

    pairs = parse_examples(raw)
    if pairs:
        return pairs

    try:
        parsed = json.loads(raw)
        if isinstance(parsed, list):
            joined = '\n'.join(str(x) for x in parsed)
            pairs = parse_examples(joined)
            if pairs:
                return pairs
    except Exception:
        pass

    try:
        parsed = ast.literal_eval(raw)
        if isinstance(parsed, list):
            joined = '\n'.join(str(x) for x in parsed)
            pairs = parse_examples(joined)
            if pairs:
                return pairs
    except Exception:
        pass

    pattern = r"Input:\s*(.*?)\s*(?:;)?\s*Output:\s*(.*?)(?=\n\s*Input:|$)"
    matches = re.findall(pattern, raw, flags=re.DOTALL | re.IGNORECASE)
    return [(a.strip(), b.strip()) for a, b in matches if a.strip() and b.strip()]


def _sanitize_question(question: str) -> str:
    q = (question or '').strip()
    if not q:
        return ''

    q = re.sub(r'^Processing condition \d+:\s*', '', q).strip()
    m = re.search(r'"question"\s*:\s*"(.*?)"', q, flags=re.DOTALL)
    if m:
        q = m.group(1).strip()

    q = q.split('\n"tests"', maxsplit=1)[0]
    q = q.split('",\n"tests"', maxsplit=1)[0]
    return q.strip(" \"'")


def load_rl_rows_from_csv(csv_path: str) -> List[dict]:
    rows: List[dict] = []
    skipped = 0

    with open(csv_path, 'r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for raw_row in reader:
            question = _sanitize_question(raw_row.get('question', ''))
            raw_tests = (raw_row.get('tests', '') or '').strip()
            raw_examples = (raw_row.get('examples', '') or '').strip()

            test_pairs = _try_parse_pairs(raw_tests)
            if not test_pairs:
                test_pairs = _try_parse_pairs(raw_examples)
            if not test_pairs:
                merged_blob = '\n'.join([raw_tests, raw_examples, raw_row.get('question', '')])
                test_pairs = _try_parse_pairs(merged_blob)

            if not question or not test_pairs:
                skipped += 1
                continue

            rows.append(
                {
                    'question': question,
                    'example': _normalize_pairs_text(test_pairs),
                    'solution': '',
                }
            )

    print(f"Loaded RL rows: {len(rows)} (skipped: {skipped}) from {csv_path}")
    return rows


def _save_sft_progress(
    progress_root: Path,
    model,
    tokenizer,
    optimizer,
    scheduler,
    epoch: int,
    best_scores: Dict[str, float],
    best_checkpoints: Dict[str, str],
    history: List[Dict],
) -> None:
    progress_root.mkdir(parents=True, exist_ok=True)
    latest_model_dir = progress_root / 'latest_model'
    latest_model_dir.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(latest_model_dir)
    tokenizer.save_pretrained(latest_model_dir)

    state = {
        'epoch': int(epoch),
        'best_scores': dict(best_scores),
        'best_checkpoints': dict(best_checkpoints),
        'history': list(history),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
    }
    state_path = progress_root / 'trainer_state.pt'
    torch.save(state, state_path)

    summary = {
        'epoch': int(epoch),
        'best_scores': {k: float(v) for k, v in best_scores.items()},
        'best_checkpoints': dict(best_checkpoints),
        'history_len': len(history),
        'state_path': str(state_path),
        'latest_model_dir': str(latest_model_dir),
    }
    with (progress_root / 'trainer_state.json').open('w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print(f"[SFT] progress saved at epoch {epoch}: {state_path}")


def _try_load_sft_progress(progress_root: Path):
    state_path = progress_root / 'trainer_state.pt'
    latest_model_dir = progress_root / 'latest_model'

    if not state_path.exists() or not (latest_model_dir / 'config.json').exists():
        return None, None

    try:
        state = torch.load(state_path, map_location='cpu')
        return state, latest_model_dir
    except Exception as e:
        print(f"[SFT] warning: failed to load progress state: {e}")
        return None, None


def train_domain_specialists(config: TrainConfig, domain_datasets: Dict[str, str], verifier, device: torch.device):
    model_factory = ModelFactory(ModelConfig(model_name=config.model_name))

    specialists_root = Path(config.output_dir) / 'specialists'
    specialists_root.mkdir(parents=True, exist_ok=True)
    progress_root = Path(config.output_dir) / 'sft_progress'
    progress_root.mkdir(parents=True, exist_ok=True)

    loaded_state = None
    resume_model_dir = None
    if config.sft_resume:
        loaded_state, resume_model_dir = _try_load_sft_progress(progress_root)

    if loaded_state is not None and resume_model_dir is not None:
        print(f"[SFT] resuming from: {resume_model_dir}")
        tokenizer = AutoTokenizer.from_pretrained(resume_model_dir)
        if tokenizer.pad_token is None:
            if tokenizer.eos_token is not None:
                tokenizer.pad_token = tokenizer.eos_token
            else:
                tokenizer.add_special_tokens({'pad_token': '[PAD]'})

        model = model_factory.load_model(str(resume_model_dir))
        model.config.pad_token_id = tokenizer.pad_token_id
        try:
            model.resize_token_embeddings(len(tokenizer))
        except Exception:
            pass
    else:
        model, tokenizer = model_factory.build()

    model.to(device)

    data_module = build_data_module(
        tokenizer=tokenizer,
        max_length=config.max_length,
        train_batch_size=config.train_batch_size,
        eval_batch_size=config.eval_batch_size,
        num_workers=config.num_workers,
        reasoning_weight=config.reasoning_weight,
        code_weight=config.code_weight,
        spec_weight=config.spec_weight,
    )

    sft_rows = data_module.load_rows_from_csv(config.sft_train_csv_path)
    if not sft_rows:
        raise ValueError(f"SFT train rows are empty: {config.sft_train_csv_path}")

    train_loader, _ = data_module.build_dataloaders_from_rows(
        train_rows=sft_rows,
        val_rows=sft_rows[:1],
    )

    domain_eval_rows: Dict[str, List[dict]] = {}
    for domain_name, csv_path in domain_datasets.items():
        rows = data_module.load_rows_from_csv(csv_path)
        if not rows:
            raise ValueError(f"Domain eval rows are empty for '{domain_name}': {csv_path}")
        domain_eval_rows[domain_name] = rows

    accum_steps = max(1, config.sft_grad_accum_steps)
    updates_per_epoch = math.ceil(len(train_loader) / accum_steps)
    total_steps = max(1, updates_per_epoch * config.sft_epochs)
    warmup_steps = int(total_steps * config.sft_warmup_ratio)

    optimizer = AdamW(model.parameters(), lr=config.sft_learning_rate, weight_decay=config.sft_weight_decay)
    scheduler = build_cosine_scheduler_with_warmup(
        optimizer=optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    engine = SFTEngine(model=model, tokenizer=tokenizer, config=config, device=device)

    best_scores: Dict[str, float] = {domain: -1.0 for domain in domain_datasets}
    best_checkpoints: Dict[str, str] = {}
    history: List[Dict] = []
    start_epoch = 1

    if loaded_state is not None:
        try:
            if 'optimizer_state' in loaded_state:
                optimizer.load_state_dict(loaded_state['optimizer_state'])
            if 'scheduler_state' in loaded_state:
                scheduler.load_state_dict(loaded_state['scheduler_state'])

            loaded_best_scores = loaded_state.get('best_scores', {}) or {}
            for domain in best_scores:
                if domain in loaded_best_scores:
                    best_scores[domain] = float(loaded_best_scores[domain])

            loaded_ckpts = loaded_state.get('best_checkpoints', {}) or {}
            best_checkpoints = {str(k): str(v) for k, v in loaded_ckpts.items()}

            loaded_history = loaded_state.get('history', []) or []
            history = list(loaded_history)

            last_epoch = int(loaded_state.get('epoch', 0))
            start_epoch = max(1, last_epoch + 1)
            print(f"[SFT] resume state loaded. Next epoch: {start_epoch}")
        except Exception as e:
            print(f"[SFT] warning: failed to restore optimizer/scheduler state: {e}")
            print('[SFT] continuing from current model weights with fresh optimizer state')

    if start_epoch > config.sft_epochs:
        print('[SFT] already completed according to saved state, skipping training')
        return best_checkpoints, best_scores, history, str(specialists_root)

    for epoch in range(start_epoch, config.sft_epochs + 1):
        train_loss = engine.train_epoch(
            loader=train_loader,
            optimizer=optimizer,
            scheduler=scheduler,
            accum_steps=accum_steps,
            max_grad_norm=config.max_grad_norm,
        )
        epoch_metrics: Dict[str, float] = {}

        print(f"\n[SFT] epoch={epoch}/{config.sft_epochs} train_loss={train_loss:.4f}")

        should_eval = (epoch % max(1, config.sft_eval_every_epochs) == 0) or (epoch == config.sft_epochs)
        if should_eval:
            for domain_name, rows in domain_eval_rows.items():
                sample_size = min(len(rows), max(1, config.sft_eval_samples_per_domain))
                sampled_rows = random.sample(rows, k=sample_size)

                score = engine.evaluate_pass_at_k(
                    rows=sampled_rows,
                    verifier=verifier,
                    k=config.pass_k,
                    n_samples=max(1, config.pass_k_samples),
                    max_rows=config.eval_max_rows,
                )
                epoch_metrics[domain_name] = score
                print(f"[{domain_name}] pass@{config.pass_k}={score:.4f} (sampled={sample_size})")

                if score > best_scores[domain_name]:
                    best_scores[domain_name] = score
                    ckpt_dir = specialists_root / f"{domain_name}_passk_{score:.4f}_epoch_{epoch}"
                    ckpt_dir.mkdir(parents=True, exist_ok=True)
                    model.save_pretrained(ckpt_dir)
                    tokenizer.save_pretrained(ckpt_dir)
                    best_checkpoints[domain_name] = str(ckpt_dir)
                    print(f"[{domain_name}] improved pass@{config.pass_k}, saved: {ckpt_dir}")
        else:
            print(f"[SFT] pass@{config.pass_k} eval skipped (every {config.sft_eval_every_epochs} epochs)")

        history.append(
            {
                'epoch': epoch,
                'train_loss': train_loss,
                'domain_pass_at_k': epoch_metrics,
            }
        )

        should_save_progress = (
            (epoch % max(1, config.sft_save_every_epochs) == 0)
            or (epoch == config.sft_epochs)
        )
        if should_save_progress:
            _save_sft_progress(
                progress_root=progress_root,
                model=model,
                tokenizer=tokenizer,
                optimizer=optimizer,
                scheduler=scheduler,
                epoch=epoch,
                best_scores=best_scores,
                best_checkpoints=best_checkpoints,
                history=history,
            )

    return best_checkpoints, best_scores, history, str(specialists_root)


def fuse_specialists(best_checkpoints: Dict[str, str], output_dir: str) -> str:
    model_paths = list(best_checkpoints.values())
    if not model_paths:
        raise ValueError('No specialist checkpoints provided for fusion.')

    merged_dir = Path(output_dir) / 'fused_model'
    merged_dir.mkdir(parents=True, exist_ok=True)

    factory = ModelFactory(ModelConfig(model_name=model_paths[0]))
    models = [factory.load_model(path) for path in model_paths]
    state_dicts = [model.state_dict() for model in models]

    fused_state = {}
    for key in state_dicts[0].keys():
        tensors = [sd[key] for sd in state_dicts]
        if tensors[0].dtype.is_floating_point:
            fused_tensor = torch.stack([t.float() for t in tensors], dim=0).mean(dim=0)
            fused_state[key] = fused_tensor.to(tensors[0].dtype)
        else:
            fused_state[key] = tensors[0]

    fused_model = factory.load_model(model_paths[0])
    fused_model.load_state_dict(fused_state)
    fused_model.save_pretrained(merged_dir)

    tokenizer = AutoTokenizer.from_pretrained(model_paths[0])
    tokenizer.save_pretrained(merged_dir)

    print(f"Fused model saved to: {merged_dir}")
    return str(merged_dir)


def evaluate_domains_with_model(model, tokenizer, config: TrainConfig, domain_datasets: Dict[str, str], verifier, device: torch.device) -> Dict[str, float]:
    data_module = build_data_module(
        tokenizer=tokenizer,
        max_length=config.max_length,
        train_batch_size=config.train_batch_size,
        eval_batch_size=config.eval_batch_size,
        num_workers=config.num_workers,
        reasoning_weight=config.reasoning_weight,
        code_weight=config.code_weight,
        spec_weight=config.spec_weight,
    )
    engine = SFTEngine(model=model, tokenizer=tokenizer, config=config, device=device)

    scores: Dict[str, float] = {}
    for domain_name, csv_path in domain_datasets.items():
        rows = data_module.load_rows_from_csv(csv_path)
        score = engine.evaluate_pass_at_k(
            rows=rows,
            verifier=verifier,
            k=config.pass_k,
            n_samples=max(1, config.pass_k_samples),
            max_rows=config.eval_max_rows,
        )
        scores[domain_name] = score
    return scores


def run_train_pipeline(config: TrainConfig) -> Dict[str, Any]:
    output_root = Path(config.output_dir)
    output_root.mkdir(parents=True, exist_ok=True)

    set_seed(config.seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    verifier = BinaryCodeVerifier()
    domain_datasets = parse_domain_csvs(config.domain_csvs)

    # SFT + fusion intentionally disabled in this notebook run.
    # best_ckpts, best_scores, sft_history, specialists_root = train_domain_specialists(...)
    # fused_model_path = fuse_specialists(...)
    sft_metrics = {
        'skipped': True,
        'reason': 'SFT/fusion disabled for MGPO-only run',
    }

    init_path = (config.rl_init_model_path or '').strip()
    if not init_path:
        raise ValueError('Set config.rl_init_model_path to your pretrained checkpoint path before running MGPO.')

    print('=== Phase: MGPO on RL dataset (SFT skipped) ===')
    rl_rows = load_rl_rows_from_csv(config.rl_train_csv_path)
    if not rl_rows:
        raise RuntimeError(f"No valid RL rows loaded from: {config.rl_train_csv_path}")

    tokenizer = AutoTokenizer.from_pretrained(init_path)
    if tokenizer.pad_token is None:
        if tokenizer.eos_token is not None:
            tokenizer.pad_token = tokenizer.eos_token
        else:
            tokenizer.add_special_tokens({'pad_token': '[PAD]'})

    load_factory = ModelFactory(ModelConfig(model_name=config.model_name))
    model = load_factory.load_model(init_path)
    model.config.pad_token_id = tokenizer.pad_token_id
    try:
        model.resize_token_embeddings(len(tokenizer))
    except Exception:
        pass

    model_vocab = int(model.get_input_embeddings().weight.shape[0])
    tokenizer_vocab = int(len(tokenizer))
    if model_vocab != tokenizer_vocab:
        model.resize_token_embeddings(tokenizer_vocab)
        model_vocab = int(model.get_input_embeddings().weight.shape[0])
    print(f"[MGPO] tokenizer_vocab={tokenizer_vocab}, model_vocab={model_vocab}")

    model_max_positions = getattr(model.config, 'n_positions', None)
    if model_max_positions is None:
        model_max_positions = getattr(model.config, 'max_position_embeddings', None)
    if isinstance(model_max_positions, int) and model_max_positions > 0:
        safe_prompt = max(8, int(model_max_positions) - int(config.rl_max_new_tokens) - 1)
        if config.max_length > safe_prompt:
            print(
                f"[MGPO] max_length reduced from {config.max_length} to {safe_prompt} "
                f"to fit model context window ({model_max_positions})"
            )
            config.max_length = safe_prompt

    model.to(device)

    grpo_config = GRPOConfig(
        learning_rate=config.rl_learning_rate,
        weight_decay=config.rl_weight_decay,
        batch_size=config.rl_batch_size,
        group_size=config.rl_group_size,
        max_new_tokens=config.rl_max_new_tokens,
        steps=config.rl_steps,
        max_length=config.max_length,
        clip_eps=config.rl_clip_eps,
        kl_beta=config.rl_kl_beta,
    )
    mgpo_trainer = MGPOTrainer(
        model=model,
        tokenizer=tokenizer,
        train_rows=rl_rows,
        device=device,
        config=grpo_config,
        verifier=verifier,
        lambda_me=config.rl_lambda_me,
    )
    rl_history = mgpo_trainer.train()
    model = mgpo_trainer.model

    final_dir = output_root / 'mgpo_final'
    final_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(final_dir)
    tokenizer.save_pretrained(final_dir)

    print('=== Final domain evaluation (after MGPO) ===')
    final_domain_scores = evaluate_domains_with_model(
        model=model,
        tokenizer=tokenizer,
        config=config,
        domain_datasets=domain_datasets,
        verifier=verifier,
        device=device,
    )

    result = {
        'config': vars(config),
        'sft': sft_metrics,
        'rl_enabled': True,
        'rl_init_model_path': init_path,
        'rl_rows_count': len(rl_rows),
        'rl_steps': len(rl_history),
        'last_rl_metrics': rl_history[-1] if rl_history else {},
        'final_model': str(final_dir),
        'final_domain_pass_at_k': final_domain_scores,
    }

    report_path = output_root / 'train_report.json'
    with open(report_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f'Train pipeline finished. Final model: {final_dir}')
    print(f'Report saved to: {report_path}')
    return result


## 8. Run Config and Start Training


In [ ]:
# Kaggle example paths (replace with your mounted dataset paths):
# /kaggle/input/<dataset-name>/reasoning_dataset.csv
# /kaggle/input/<dataset-name>/alg_tasks.csv
# /kaggle/input/<dataset-name>/math_tasks.csv
# /kaggle/input/<dataset-name>/struct_tasks.csv
# /kaggle/input/<dataset-name>/rl_tasks_from_llm_2kplus.csv

config = TrainConfig(
    # Required for MGPO-only run: path to pretrained checkpoint (SFT result).
    rl_init_model_path='weights/gpt2_weigths_30_epochs/artifacts_easy_gpt2/best_model',
    sft_train_csv_path='data/reasoning_dataset.csv',
    domain_csvs='alg:data/alg_tasks.csv,math:data/math_tasks.csv,struct:data/struct_tasks.csv',
    rl_train_csv_path='data/rl_tasks_from_llm_2kplus.csv',

    model_name='gpt2',
    output_dir='artifacts_train_gpt2',
    max_length=768,

    sft_epochs=2,
    sft_learning_rate=5e-5,
    sft_weight_decay=0.01,
    sft_warmup_ratio=0.05,
    sft_grad_accum_steps=1,
    max_grad_norm=1.0,
    sft_eval_every_epochs=3,
    sft_eval_samples_per_domain=20,
    sft_save_every_epochs=1,
    sft_resume=True,

    pass_k=3,
    pass_k_samples=8,
    eval_max_new_tokens=256,
    eval_temperature=0.8,
    eval_top_p=0.95,
    eval_max_rows=0,

    rl_steps=100,
    rl_learning_rate=1e-5,
    rl_weight_decay=0.0,
    rl_batch_size=4,
    rl_group_size=8,
    rl_clip_eps=0.2,
    rl_kl_beta=0.0,
    rl_lambda_me=4.0,
    rl_max_new_tokens=128,

    seed=42,
    train_batch_size=2,
    eval_batch_size=2,
    num_workers=0,
    reasoning_weight=1.0,
    code_weight=1.0,
    spec_weight=1.0,
)

result = run_train_pipeline(config)
result
